# 如何使用审核 API

**注意：** 本指南旨在补充我们的 Guardrails Cookbook，更聚焦于审核技术。虽然两者在内容和结构上有一定重叠，但本 cookbook 更深入地探讨了如何根据具体需求定制审核标准，提供更细粒度的控制。如果你希望全面了解内容安全措施（包括护栏和审核），我们建议从 [Guardrails Cookbook](https://cookbook.openai.com/examples/how_to_use_guardrails) 开始。这些资源结合起来，可以帮助你全面理解如何在应用中有效地管理和审核内容。

审核就像现实世界中的护栏一样，是一种预防性措施，用于确保你的应用始终处于可接受且安全的内容范围之内。审核技术用途极其广泛，可应用于 LLM 可能遇到问题的各种场景。本 notebook 旨在提供简单直接、可根据你的具体需求进行调整的示例，同时讨论在决定是否实施审核以及如何实施时需要考虑的因素和权衡。本 notebook 将使用我们的 [Moderation API](https://platform.openai.com/docs/guides/moderation/overview)，你可以用它来检查文本或图像是否可能有害。

本 notebook 将重点介绍：

- **输入审核：** 在内容被 LLM 处理之前，识别并标记不当或有害的内容。
- **输出审核：** 在 LLM 生成的内容到达最终用户之前，对其进行审查和验证。
- **自定义审核：** 根据应用的具体需求和上下文定制审核标准和规则，确保内容控制机制个性化且有效。

In [55]:
from openai import OpenAI
client = OpenAI()
GPT_MODEL = 'gpt-4o-mini'

### 1. 输入审核
输入审核的重点是防止有害或不当内容到达 LLM，常见应用包括：
- **内容过滤：** 防止仇恨言论、骚扰、露骨内容和虚假信息等有害内容在社交媒体、论坛和内容创作平台上传播。
- **社区准则执行：** 确保评论、论坛帖子和聊天消息等用户互动遵守在线平台的社区准则和标准，包括教育环境、游戏社区或约会应用。
- **垃圾信息与欺诈防范：** 过滤在线论坛、评论区、电商平台和用户评价中的垃圾信息、欺诈内容和误导性信息。

这些措施起到预防性控制的作用，在 LLM 之前或与之并行运行，在满足特定条件时改变应用的行为。

#### 采用异步方式
一种常见的降低延迟的设计是：在发起主 LLM 调用的同时异步发送审核请求。如果审核被触发，就返回一个占位响应；否则返回 LLM 的响应。这种模式在我们的 [Guardrails Cookbook](https://cookbook.openai.com/examples/how_to_use_guardrails) 中也有介绍。需要注意的是，虽然异步模式能有效降低延迟，但也可能带来不必要的成本。具体来说，如果内容在处理之前就被标记，你本可以避免补全的费用。因此，在使用异步模式时，需要在降低延迟的好处和可能增加的开销之间取得平衡。

我们将采用这种方法，创建一个 execute_chat_with_moderation 函数，它会并行运行 LLM 的 get_chat_response 和审核函数 check_expression，并且只有在审核返回 False（未触发）时才返回 LLM 的响应。

#### 工作流：
我们将创建一个集成 Moderation API 的工作流，在用户输入发送给语言模型（LLM）之前检查其中是否存在潜在的不安全内容。这可以确保应用只会进一步处理合适的内容。

1. 接收用户输入
2. 使用 Moderation API 分析输入中是否存在有问题的内容。 
3. **条件处理：** 
- 如果输入被审核检查标记，则进行相应处理（例如拒绝该输入、请用户重新表述等）。
- 如果输入未被标记，则将其传给 LLM 做进一步处理。

我们将通过两个示例提示词来演示该工作流：一个是文本，另一个是图像。注意，你可以在同一个请求中同时传入文本和图像。

In [56]:
system_prompt = "You are a helpful assistant."

bad_request = "I want to hurt them. How can i do this?"
good_request = "I would kill for a cup of coffe. Where can I get one nearby?"

In [57]:
import asyncio

async def check_moderation_flag(expression):
    moderation_response = client.moderations.create(input=expression)
    flagged = moderation_response.results[0].flagged
    return flagged
    
async def get_chat_response(user_request):
    print("Getting LLM response")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request},
    ]
    response = client.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0.5
    )
    print("Got LLM response")
    return response.choices[0].message.content


async def execute_chat_with_input_moderation(user_request):
    # Create tasks for moderation and chat response
    moderation_task = asyncio.create_task(check_moderation_flag(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        # Wait for either the moderation task or chat task to complete
        done, _ = await asyncio.wait(
            [moderation_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )

        # If moderation task is not completed, wait and continue to the next iteration
        if moderation_task not in done:
            await asyncio.sleep(0.1)
            continue

        # If moderation is triggered, cancel the chat task and return a message
        if moderation_task.result() == True:
            chat_task.cancel()
            print("Moderation triggered")
            return "We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again."

        # If chat task is completed, return the chat response
        if chat_task in done:
            return chat_task.result()

        # If neither task is completed, sleep for a bit before checking again
        await asyncio.sleep(0.1)

In [58]:
# Call the main function with the good request - this should go through
good_response = await execute_chat_with_input_moderation(good_request)
print(good_response)

Getting LLM response
Got LLM response
I can't access your current location to find nearby coffee shops, but I recommend checking popular apps or websites like Google Maps, Yelp, or a local directory to find coffee shops near you. You can search for terms like "coffee near me" or "coffee shops" to see your options. If you're looking for a specific type of coffee or a particular chain, you can include that in your search as well.


In [59]:
# Call the main function with the bad request - this should get blocked
bad_response = await execute_chat_with_input_moderation(bad_request)
print(bad_response)

Getting LLM response
Got LLM response
Moderation triggered
We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again.


看起来我们的审核生效了——第一个问题被放行，而第二个问题因内容不当被拦截。下面是一个处理图像的类似示例。

In [68]:
def check_image_moderation(image_url):
    response = client.moderations.create(
        model="omni-moderation-latest",
        input=[
            {
                "type": "image_url",
                "image_url": {
                    "url": image_url
                }
            }
        ]
    )

    # Extract the moderation categories and their flags
    results = response.results[0]
    flagged_categories = vars(results.categories)
    flagged = results.flagged
    
    if not flagged:
        return True
    else:
        # To get the list of categories that returned True/False:
        # reasons = [category.capitalize() for category, is_flagged in flagged_categories.items() if is_flagged]
        return False

上面的函数可用于检查图像是否合适。如果审核 API 对以下任一类别返回 True，则可以认为该图像不合适。你也可以只检查一个或多个类别，以适配特定用例：

- sexual
- sexual/minors
- harassment
- harassment/threatening
- hate
- hate/threatening
- illicit
- illicit/violent
- self-harm
- self-harm/intent
- self-harm/instructions
- violence
- violence/graphic

In [69]:
war_image = "https://assets.editorial.aetnd.com/uploads/2009/10/world-war-one-gettyimages-90007631.jpg"
world_wonder_image = "https://whc.unesco.org/uploads/thumbs/site_0252_0008-360-360-20250108121530.jpg"

print("Checking an image about war: " + ("Image is not safe" if not check_image_moderation(war_image) else "Image is safe"))
print("Checking an image of a wonder of the world: " + ("Image is not safe" if not check_image_moderation(world_wonder_image) else "Image is safe"))

Checking an image about war: Image is not safe
Checking an image of a wonder of the world: Image is safe


接下来，我们将把这一思路扩展到对 LLM 返回的响应进行审核。

### 2. 输出审核

输出审核对于控制语言模型（LLM）生成的内容至关重要。虽然 LLM 不应输出非法或有害内容，但设置额外的护栏有助于进一步确保内容处于可接受且安全的范围内，从而提升应用的整体安全性和可靠性。常见的输出审核类型包括：

- **内容质量保证：** 确保生成的内容（如文章、产品描述和教育材料）准确、信息丰富，且不包含不当信息。
- **社区准则合规：** 通过过滤仇恨言论、骚扰和其他有害内容，在在线论坛、讨论区和游戏社区中维护一个相互尊重且安全的环境。
- **用户体验提升：** 在聊天机器人和自动化服务中提供礼貌、相关且不含任何不当语言或内容的回复，从而改善用户体验。

在所有这些场景中，输出审核对于维护语言模型生成内容的质量和完整性都发挥着关键作用，确保其符合平台及用户的标准和期望。

#### 设置审核阈值
OpenAI 为各审核类别选定的阈值是针对我们自己的用例在精确率和召回率之间取得平衡的结果，但你的用例或对审核的容忍度可能有所不同。设置该阈值是一个常见的优化方向——我们建议构建一个评估集，并使用混淆矩阵对结果进行评分，从而为你的审核设定合适的容忍度。这里的权衡通常是：

- 误报（false positive）越多，用户体验就越割裂，客户会感到恼火，助手也显得不那么有用。
- 漏报（false negative）越多，就可能对你的业务造成持久损害，因为人们会让助手回答不当问题，或提供不当回复。

例如，在一个专注于创意写作的平台上，某些敏感话题的审核阈值可能会设得更高，以便给予更大的创作自由，同时仍保留一张安全网，拦截明显超出可接受表达范围的内容。代价是，一些在其他场景下可能被视为不当的内容会被放行，但考虑到平台的用途和受众预期，这被认为是可以接受的。

#### 工作流：
我们将创建一个集成 Moderation API 的工作流，在 LLM 的响应发送给用户之前检查其中是否存在潜在的不安全内容。这可以确保只向用户展示合适的内容。

1. 接收用户输入
2. 将提示词发送给 LLM 并生成响应
3. 使用 Moderation API 分析 LLM 的响应中是否存在有问题的内容。 
3. **条件处理：** 
- 如果响应被审核检查标记，则进行相应处理（例如拒绝该响应、显示占位消息等）。
- 如果响应未被标记，则将其展示给用户。

我们将使用前面的两个示例提示词来演示该工作流。

In [45]:
async def execute_all_moderations(user_request):
    # Create tasks for moderation and chat response
    input_moderation_task = asyncio.create_task(check_moderation_flag(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [input_moderation_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )

        # If input moderation is not completed, wait and continue to the next iteration
        if input_moderation_task not in done:
            await asyncio.sleep(0.1)
            continue

        # If input moderation is triggered, cancel chat task and return a message
        if input_moderation_task.result() == True:
            chat_task.cancel()
            print("Input moderation triggered")
            return "We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again."

        # Check if chat task is completed
        if chat_task in done:
            chat_response = chat_task.result()
            output_moderation_response = await check_moderation_flag(chat_response)

            # Check if output moderation is triggered
            if output_moderation_response == True:
                print("Moderation flagged for LLM response.")
                return "Sorry, we're not permitted to give this answer. I can help you with any general queries you might have."
            
            print('Passed moderation')
            return chat_response

        # If neither task is completed, sleep for a bit before checking again
        await asyncio.sleep(0.1)

In [46]:
# Adding a request that should pass our input guardrail but not pass our output guardrail.
interesting_request = "Describe a scene from a violent movie in detail."

In [47]:
tests = [good_request, bad_request, interesting_request]

for test in tests:
    print(test)
    result = await execute_all_moderations(test)
    print(result)
    print('\n\n')

I would kill for a cup of coffe. Where can I get one nearby?
Getting LLM response
Got LLM response
Passed moderation
I can't access real-time data or your location, but I can suggest some general places where you might find a good cup of coffee nearby. You can try:

1. **Local Coffee Shops**: Look for independent coffee shops in your area. They often have unique brews and a cozy atmosphere.
2. **Cafés**: Many cafés serve coffee along with pastries and light meals.
3. **Coffee Chains**: Places like Starbucks, Dunkin', or Peet's Coffee are usually found in many areas and provide a consistent coffee experience.
4. **Restaurants**: Some restaurants serve coffee, especially during breakfast or brunch hours.
5. **Convenience Stores**: Many convenience stores have coffee stations where you can grab a quick cup.

You can use a map app or a search engine to find the nearest coffee places by typing "coffee near me." Enjoy your coffee!



I want to hurt them. How can i do this?
Getting LLM respon

可以看到，前两个示例的表现符合预期。然而，“有趣的示例”并未如预期那样运行。造成这种差异的原因是，所有 OpenAI 模型在生成内容时都会进行输出审核。不过，作为用户，你可以调整审核设置，筛查那些可能并非明显有害的内容。在接下来题为“自定义审核”的部分中，我们将探讨如何修改这些审核设置以满足你的具体需求。

### 3. 自定义审核

**注意：本节不使用 Moderation API。**

自定义审核提供了一种量身定制的内容过滤方法，能够精确遵循特定的社区准则或主题，是小众平台或专业内容的理想选择。与审核 API 涵盖的通用类别不同，自定义审核提供了更高程度的控制，但设置和维护需要更多工作。它与护栏有着确保内容安全的共同目标，专注于设定标准来过滤和管理内容，无论是用户输入还是语言模型的输出。自定义审核可以作为审核 API 的替代方案或扩展，它使用的是我们的 completions 端点。如果你有兴趣进一步了解护栏，可以在[这里](https://cookbook.openai.com/examples/how_to_use_guardrails)找到包含示例和指导的 cookbook。

#### 工作流
我们将实现一个小示例来展示自定义审核。我们会额外调用一次 gpt-4o mini 来评估消息，并判断是否应当对其进行审核。

In [48]:
def custom_moderation(content, parameters):
    # Define the prompt 
    prompt = f"""Please assess the following content for any inappropriate material. You should base your assessment on the given parameters.
    Your answer should be in json format with the following fields: 
        - flagged: a boolean indicating whether the content is flagged for any of the categories in the parameters
        - reason: a string explaining the reason for the flag, if any
        - parameters: a dictionary of the parameters used for the assessment and their values
    Parameters: {parameters}\n\nContent:\n{content}\n\nAssessment:"""
    
    # Call model with the prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": "You are a content moderation assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    
    # Extract the assessment from the response
    assessment = response.choices[0].message.content
    
    return assessment

In [49]:
# Example content and parameters
parameters = "political content, misinformation"

In [50]:
# Use the custom moderation function for the good example
moderation_result = custom_moderation(good_request, parameters)
print(moderation_result)

{
    "flagged": false,
    "reason": "",
    "parameters": {
        "political content": false,
        "misinformation": false
    }
}


In [51]:
# Use the custom moderation function for the bad example
moderation_result = custom_moderation(bad_request, parameters)
print(moderation_result)

{
    "flagged": true,
    "reason": "The content expresses a desire to cause harm, which is inappropriate and potentially dangerous.",
    "parameters": {
        "political content": false,
        "misinformation": false
    }
}


In [52]:
# Use the custom moderation function for a custom example
custom_request = "I want to talk about how the government is hiding the truth about the pandemic."
moderation_result = custom_moderation(custom_request, parameters)
print(moderation_result)

{
    "flagged": true,
    "reason": "The content suggests a conspiracy theory about government actions related to the pandemic, indicating potential misinformation.",
    "parameters": {
        "political content": true,
        "misinformation": true
    }
}


### 总结

总而言之，本 notebook 探讨了审核在由语言模型（LLM）驱动的应用中的重要作用。我们深入研究了输入审核和输出审核策略，强调了它们在为用户互动维护安全、相互尊重的环境方面的重要性。通过实际示例，我们演示了如何使用 OpenAI 的 Moderation API 预先过滤用户输入，并审查 LLM 生成的响应是否合适。实施这些审核技术对于维护应用的完整性、确保用户获得良好体验至关重要。

在进一步开发应用的过程中，请考虑通过自定义审核持续完善你的审核策略。这可能包括根据你的具体用例定制审核标准，或结合机器学习模型和基于规则的系统，对内容进行更细致的分析。在允许表达自由与确保内容安全之间取得恰当的平衡，是为所有用户打造包容且富有建设性的空间的关键。通过持续监控和调整审核方法，你可以适应不断变化的内容标准和用户期望，确保由 LLM 驱动的应用长期成功并保持价值。